> **How schema changes work here.** There is no in-place `ALTER TABLE` and no
> `/admin/schemas/.../evolve` API — that framework was removed (`854c559b`). A
> collection's columns are declared in its `ItemsSchema` config and materialised
> **once** at provisioning via `CREATE TABLE IF NOT EXISTS`. You add a field by
> evolving the *declared* schema (mutable until the table has rows); adding a
> column after data exists requires a fresh (re)provision, never an app-issued
> `ALTER`. See `docs/components/schema_evolution.md` and
> `docs/components/items_schema.md`.

# UC-CAT-3 — Collection Schema: Declare Up-Front, Evolve via Reprovision

**Persona:** Data Engineer

**Goal:** Define a collection's field schema declaratively, add a field while the
collection is still empty, and understand the supported path for changing the
schema after data exists.

1. Create a test collection with the PostgreSQL driver configured
2. Declare the schema via the `ItemsSchema` config (`fields`, `strict_unknown_fields`, `default_access`)
3. Evolve the declared schema by adding a field (pre-materialisation)
4. Verify the field surfaces in OGC queryables
5. Understand the after-data rule: reprovision, never in-place `ALTER`
6. Clean up the test collection

**Prerequisites:**
- A catalog already exists (or this notebook creates an ephemeral one)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- `driver:postgresql` installed on the platform

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", "") or os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")
# Use a notebook-unique catalog so the notebook is fully self-contained
CATALOG_ID = f"schema-evo-test-{uuid.uuid4().hex[:8]}"

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=60.0)
print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}")

# Create the test catalog
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "Schema Evolution Test Catalog",
    "description": "Ephemeral catalog for schema evolution demo.",
    "stac_version": "1.0.0",
}
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
assert r.status_code == 201, f"Failed to create catalog: {r.status_code}: {r.text}"
print(f"Created catalog: {CATALOG_ID}")

## Setup — Create test collection with PostgreSQL driver

Before schema evolution, we need a collection with PostgreSQL storage configured.
This cell creates it automatically so the notebook is self-contained.

In [ ]:
import uuid

COLLECTION_ID = f"schema-evolution-test-{uuid.uuid4().hex[:8]}"

# Ensure the catalog has a default routing config for collection creation.
catalog_defaults = {
    "configs": {
        "items_postgresql_driver_config": {"enabled": True, "collection_type": "VECTOR"},
        "collection_routing_config": {
            "enabled": True,
            "operations": {
                "WRITE": [{"driver_ref": "items_postgresql_driver", "hints": [], "on_failure": "fatal"}],
                "READ":  [{"driver_ref": "items_postgresql_driver", "hints": [], "on_failure": "fatal"}],
            },
        },
    }
}
r_bulk = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps(catalog_defaults), timeout=60.0)
print(f"Catalog defaults: {r_bulk.status_code}")

# Create the test collection
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "Schema Evolution Test Collection",
    "description": "Test collection for schema evolution demo.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}
r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections", content=json.dumps(collection_payload))
assert r.status_code == 201, f"Failed to create collection: {r.status_code}: {r.text}"
print(f"Created test collection: {COLLECTION_ID}")

## Step 0a — Configure the collection schema declaratively (`ItemsSchema`)

Before evolving a live schema, the supported path for *defining* the schema in
the first place is to PUT an `ItemsSchema` config at collection scope. The
config waterfall lets you:

* declare typed `fields` (drives DDL columns + service-layer validation);
* declare write-time identity, dedup, and computed fields on the companion
  `ItemsWritePolicy` config (PR #961 moved these off `ItemsSchema.constraints`,
  see `ItemsWritePolicy.identity` and `ItemsWritePolicy.compute`);
* flip `strict_unknown_fields=True` to reject writes whose properties aren't in
  `fields` (HTTP 422 with the offending field list);
* flip `default_access="fast"` to lift every declared field onto
  a native PG column on the attributes sidecar (per-field indexes, ANALYZE
  stats, column-store plans).

These flags can be set per-collection without touching the platform or catalog
defaults — the resolved view at
`GET /configs/catalogs/{cat}/collections/{col}?resolved=true` shows where each
value came from via the `inherited` tree.


In [ ]:
# 1) Read current ItemsSchema at collection scope (will be the code default
#    until we PUT an override).
r = client.get(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    f"/plugins/items_schema",
)
print("BEFORE:", r.status_code)
print(json.dumps(r.json(), indent=2)[:600])


In [ ]:
# 2) PUT a typed schema with two declared fields and strict-unknown enforcement.
#    Identity / dedup rules live on ItemsWritePolicy now (PR #961), not on
#    ItemsSchema.constraints.
items_schema = {
    "level": "ITEM",
    "fields": {
        "feature_id": {"data_type": "text"},
        "cloud_cover_pct": {"data_type": "numeric"},
    },
    "strict_unknown_fields": True,
    "default_access": "fast",
}

r = client.put(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    f"/plugins/items_schema",
    content=json.dumps(items_schema),
)
print("PUT:", r.status_code, r.text[:400])
assert r.status_code in (200, 201), r.text


In [ ]:
# 3) Read it back via the per-plugin GET — fastest sanity check.
r = client.get(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    f"/plugins/items_schema",
)
print("AFTER per-plugin GET:", r.status_code)
print(json.dumps(r.json(), indent=2)[:800])


In [ ]:
# 4) Confirm the override surfaces in the composed/resolved collection view.
#    The `inherited` tree carries `{"source": "collection"}` at the same address
#    that produced the payload — so you can see *where* each value came from.
r = client.get(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}",
    params={"resolved": "true", "meta": "field", "include": "scope", "strict": "true"},
)
assert r.status_code == 200, r.text
body = r.json()
schema_node = (
    body.get("configs", {})
        .get("collection", {})
        .get("items", {})
        .get("schema", {})
        .get("items_schema")
)
print("composed items_schema:")
print(json.dumps(schema_node, indent=2)[:800])


### Note — sidecar defaults look empty until materialisation

`ItemsPostgresqlDriverConfig.sidecars` defaults to `[]`. That is **not** a
missing config: the PG driver resolves the effective sidecar list lazily at
`ensure_storage()` time via `_effective_sidecars(...)`, then persists the
resolved snapshot back onto this field. Before the first feature is ingested
the stored value is `[]`; after materialisation it carries
`geometries`/`attributes` plus extension-registered sidecars
(`item_metadata`, `stac_metadata`, …). Because the field is `Immutable`, any
explicit override has to be PUT *before* the table is created.


## Step 0 — Guard: confirm no reindex is running

Applying a schema evolution while an active reindex is in progress can produce inconsistent
index mappings. Always check before evolving.

In [ ]:
# Query the catalog task log for active REINDEX events
r = client.get(
    f"/logs/catalogs/{CATALOG_ID}/logs",
    params={"event_type": "REINDEX", "level": "INFO", "limit": 10},
)
print(r.status_code, r.text[:500])
# 200 expected; 404 is acceptable if no log backend is configured
assert r.status_code in (200, 404), (
    f"Unexpected status {r.status_code}: {r.text}"
)

if r.status_code == 200:
    log_entries = r.json()
    active_reindex = [
        e for e in (log_entries if isinstance(log_entries, list) else log_entries.get("entries", []))
        if e.get("status") in ("running", "accepted", "RUNNING", "ACCEPTED")
    ]
    assert len(active_reindex) == 0, (
        f"Active reindex detected — do not evolve schema while reindex is running: {active_reindex}"
    )
    print(f"No active reindex found ({len(log_entries if isinstance(log_entries, list) else log_entries.get('entries', []))} recent log entries checked). Safe to proceed.")
else:
    print("Log backend not available (404). Proceeding without reindex guard — verify manually.")

## Step 1 — Evolve the declared schema (before data exists)

Step 0a declared `feature_id` and `cloud_cover_pct`. To add another field —
`sensor_id` — re-declare the `ItemsSchema`. While the collection has **no
rows** the schema is still mutable, so the change is accepted and the next
provisioning materialises the new column. (A JSON merge-patch with
`Content-Type: application/merge-patch+json` works for incremental edits too;
we re-PUT the full schema here for clarity.)

In [ ]:
# Re-declare the schema with one additional field (sensor_id).
items_schema_v2 = {
    "level": "ITEM",
    "fields": {
        "feature_id": {"data_type": "text"},
        "cloud_cover_pct": {"data_type": "numeric"},
        "sensor_id": {"data_type": "text"},  # newly added field
    },
    "strict_unknown_fields": True,
    "default_access": "fast",
}

r = client.put(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    f"/plugins/items_schema",
    content=json.dumps(items_schema_v2),
)
print("PUT:", r.status_code, r.text[:400])
assert r.status_code in (200, 201), r.text

# Read back — the declared schema now carries sensor_id.
r = client.get(
    f"/catalog/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}"
    f"/plugins/items_schema",
)
fields = r.json().get("fields", {})
print("declared fields:", sorted(fields))
assert "sensor_id" in fields, f"sensor_id missing from declared schema: {sorted(fields)}"
print("sensor_id is now part of the declared schema.")

## Step 2 — Verify via queryables

The OGC API Queryables endpoint reflects the collection's resolved schema.
Declared fields with query access surface here as filterable CQL2 properties.

In [ ]:
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables"
)
print(r.status_code, r.text[:800])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

queryable_names = list(r.json().get("properties", {}).keys())
print(f"Queryable properties ({len(queryable_names)}): {queryable_names}")
for f in ("cloud_cover_pct", "sensor_id"):
    mark = "\u2713" if f in queryable_names else "—"
    print(f"  {mark} {f}")

## Adding a column after data exists — reprovision, never `ALTER`

Once the collection's physical table has rows, the declared schema is frozen
for shape-changing edits. The application **never** issues `ALTER TABLE … ADD/
DROP/RENAME COLUMN` against a materialised table — that is a hard invariant.

- **Drift is reconciled read-only.** If a config advertises a column the
  physical table lacks, `ensure_storage` reconciles the *config down to
  physical reality* (drops the unbacked entry with a WARN) so reads don't
  500 — it never mutates the table.
- **To truly add a column,** (re)provision the collection: create it (or a new
  version) with the updated `ItemsSchema`, then re-ingest. There is no
  in-place evolution path and no backup/restore evolve API.

See `docs/components/schema_evolution.md` (the model) and
`docs/architecture/migrations.md` (schema establishment).

In [ ]:
print("Session will be closed after teardown.")

## Teardown

Clean up the test collection created in the setup cell above.

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(f"Collection delete: {r.status_code}")
# 204 = deleted; 404 = already gone (notebook re-run); both are acceptable
assert r.status_code in (204, 404), f"Unexpected status: {r.status_code}: {r.text}"
print(f"Test collection {COLLECTION_ID} cleaned up.")

r2 = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
assert r2.status_code == 204, f"Failed to delete catalog: {r2.status_code}: {r2.text}"
print(f"Test catalog {CATALOG_ID} deleted.")
client.close()
print("Session closed.")